## Explore Dataset

In [ ]:
from ipywidgets import interact, IntSlider
import importlib
import geopandas as gpd
import dotenv
import os
import pandas as pd
import shapely
from google.cloud import storage
import numpy as np
from utils import csoRisk
import RiskMap
import matplotlib.pyplot as plt
from collections.abc import Callable
import momepy as mp

import utils.graph
importlib.reload(utils.graph)

importlib.reload(RiskMap)

In [ ]:
dotenv.load_dotenv(override=True)

## Obtaining data

Keeping all the data in the EPSG 3857 coordinate referencing system (CRS) to work better with distances

### CSO Data

In [ ]:
# geo_JSON_url = "https://services-eu1.arcgis.com/MSNNjkZ51iVh8yBj/arcgis/rest/services/Northumbrian_Water_Storm_Overflow_Activity_2_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
# cso: gpd.GeoDataFrame = gpd.read_file(geo_JSON_url)
# cso_all = cso.to_crs(epsg=3857)

In [ ]:
INPUT_PATH = os.getenv("CSO_DATA_PATH")
if INPUT_PATH is None:
    raise ValueError("Input Path not set")
bucketName = INPUT_PATH.split("/")[2]
path_in_bucket = '/'.join(INPUT_PATH.split("/")[3:])
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucketName)
geojson_filenames = ["gs://"+bucketName+'/' +
                     fname.name for fname in bucket.list_blobs(prefix=path_in_bucket) if fname.name.endswith('.geojson')]
cso = gpd.read_file(geojson_filenames[-1])
cso_all = cso.to_crs(epsg=3857)

### SSSI data

In [ ]:
SSSI_PATH = os.getenv("SSSI_DATA_PATH")
sssri_all = gpd.read_file(SSSI_PATH).to_crs(epsg=3857)

### Bathing Water data

In [ ]:
BATHW_PATH = os.getenv("BATHW_DATA_PATH")
bathw_all = gpd.read_file(BATHW_PATH).to_crs(epsg=3857)

### WwTW data

In [ ]:
WWTW_PATH = os.getenv("WWTW_DATA_PATH")
wwtw_all = gpd.read_parquet(WWTW_PATH)

### River data

In [ ]:
RIVERS_PATH = os.getenv("RIVERS_PATH")
rivers = gpd.read_file(RIVERS_PATH)
rivers = rivers.to_crs(epsg=3857)
rivers_all = rivers.copy()

### Clip to cso bounds

In [ ]:
rivers = rivers_all.clip(cso_all.total_bounds).explode()
cso = cso_all

### Clip to Catchment Data

In [ ]:
cmnt = gpd.read_file(
    "https://environment.data.gov.uk/catchment-planning/OperationalCatchment/3052.geojson"
)  # Browney
# cmnt = gpd.read_file(
#     "https://environment.data.gov.uk/catchment-planning/ManagementCatchment/3110.geojson"
# )
cmnt.to_crs(cso_all.crs, inplace=True)

In [ ]:
rivers = gpd.clip(rivers_all, cmnt)
# Remove all rivers that aren't LineStrings after Clipping
rivers = rivers[rivers.geometry.apply(
    lambda x: isinstance(x, shapely.LineString))]
cso = gpd.clip(cso_all, cmnt)

In [ ]:
wwtw = gpd.clip(wwtw_all, cmnt)

In [ ]:
bathw = bathw_all.clip(cmnt.total_bounds)
sssri = gpd.clip(sssri_all, cmnt)


def collection(x):
    if isinstance(x, shapely.GeometryCollection):
        return list(x.geoms)
    else:
        return [x]


sssri["geometry2"] = sssri.geometry.apply(
    lambda x: [poly for poly in collection(x) if isinstance(poly, shapely.Polygon)])
sssri = sssri.explode("geometry2").set_geometry(
    "geometry2").set_crs(sssri_all.crs)

### Pre Explore

In [ ]:
m = cmnt.explore(highlight=False,
                 popup=False,
                 tooltip=False,
                 color="grey")
rivers.explore(m=m)
cso.explore(m=m, marker_kwds={"radius": 5}, color="red")
wwtw.set_geometry("EFFLUENT_LOC").explore(
    m=m, marker_kwds={"radius": 5}, color="purple")

## Construct Risk Info Map

In [ ]:
RIVER_RESOLUTION: float = 500
MAX_OUTFALL_DIST_FROM_RIVER: float = 700
MAX_RISK_DISTANCE: float = 10_000

In [ ]:
# Generate Graph from Rivers
riskmap = RiskMap.RiskMap(rivers, "OBJECTID", RIVER_RESOLUTION, directed=True)
riskmap.construct_river_graph()

In [ ]:
# Add CSO information
outfallLocations = riskmap.add_point_info_to_map(
    cso["id"], cso["geometry"], 'csoInfo', MAX_RISK_DISTANCE, MAX_OUTFALL_DIST_FROM_RIVER)
cso = cso.assign(outfallLocation=outfallLocations)

In [ ]:
# Add WwTW information
outfallLocations = riskmap.add_point_info_to_map(
    wwtw["PERMIT_NUMBER"], wwtw["EFFLUENT_LOC"], 'wwtwInfo',
    MAX_RISK_DISTANCE, MAX_OUTFALL_DIST_FROM_RIVER
)
wwtw = wwtw.assign(outfallLocation=outfallLocations)

In [ ]:
def linear_dist_decrease(d: float) -> float:
    return (MAX_RISK_DISTANCE-d)/MAX_RISK_DISTANCE

In [ ]:
# Construct and get the GDF
riskmap.add_all_risk(
    [
        {
            "info_col": "csoInfo",
            "risk_name": "csoRisk",
            "risk_method": csoRisk.add_duration_risk,
            "distance_scaling": linear_dist_decrease,
        },
        {
            "info_col": "wwtwInfo",
            "risk_name": "wwtwRisk",
            "risk_method": lambda x: 100,
            "distance_scaling": linear_dist_decrease,
        },
    ]
)
riskmap.construct_risk_map()
river_points = riskmap.get_risk_map()

## Explore

In [ ]:
m = cmnt.explore(
    highlight=False,
    tooltip=False,
    popup=False,
    color="grey",
)
rivers.explore(
    m=m,
    color="blue",
)
river_points.explore(
    m=m,
    column="Risk",
    cmap='viridis',
    marker_kwds={"radius": 5},
)
wwtw.set_geometry("outfallLocation").explore(
    m=m,
    color="dark blue",
    marker_kwds={"radius": 5}
)
# bathw.explore(
#     m=m,
#     color = "yellow",
#     marker_kwds={"radius":5}
# )
# sssri.explore(
#     m=m,
#     color="green",
#     marker_kwds={"radius": 5}
# )
# cso.explore(
#     m=m,
#     color="red",
#     marker_kwds={"radius": 5}
# )
cso.set_geometry("outfallLocation").explore(
    m=m,
    color="pink",
    marker_kwds={"radius": 8}
)

## Optimize Placement of Sensors

In [ ]:
N_SENSORS = 10

### Greedy

In [ ]:
history = riskmap.optimization_greedy(
    N_SENSORS,
    [
        {
            "info_col": "csoInfo",
            "risk_name": "csoRisk",
            "risk_method": csoRisk.add_duration_risk,
            "distance_scaling": linear_dist_decrease,
        }
    ],
    2000,
    hist=True,
)

In [ ]:
def plot(x):
    m = rivers.explore()

    if isinstance(history, tuple):
        history[0].explore(m=m, marker_kwds={"radius": 5}, color="red")
        riskmap.get_risk_map().explore(m=m, column=history[1])

    else:
        if len(history[x][0]) > 0:
            history[x][0].explore(m=m, marker_kwds={"radius": 5}, color="red")
        riskmap.get_risk_map().explore(m=m, column=history[x][1])
    return m


interact(plot, x=IntSlider(min=0, max=len(history)-1, step=1))

### Platypus

In [ ]:
from platypus import NSGAII, Problem, Direction, nondominated, unique, Binary, Constraint

In [ ]:
class SensorProblem(Problem):

    def __init__(self, riskmap: RiskMap.RiskMap, riskList: list[dict], max_dist_sensor: float):
        l = len(riskmap.get_risk_map())
        super().__init__(1, 1, 1)
        self.riskmap = riskmap
        self.max_dist_sensor = max_dist_sensor
        self.riskList = riskList

        self.types[0] = Binary(l)
        self.constraints[0] = Constraint("<=", N_SENSORS)
        self.directions[0] = Direction.MINIMIZE

    def evaluate(self, solution):
        # Get the sensors points
        sensorLocs = self.riskmap.get_risk_map()[solution.variables[0]]

        # Add ids covered by the sensor
        riskList2 = []
        for item in self.riskList:
            riskList2.append(item.copy())
            riskList2[-1]["ids_to_ignore"] = [thing[0] for row in sensorLocs[item["info_col"]].to_list(
            ) for thing in row if thing[1] < self.max_dist_sensor]
        self.riskmap.add_all_risk(riskList2)
        new_risk = self.riskmap.get_risk_map()

        # Reset riskmap
        self.riskmap.add_all_risk(self.riskList)

        # Set as solution objective
        solution.objectives[0] = new_risk["Risk"].sum()/len(new_risk)
        solution.constraints[0] = sum(solution.variables[0])

        # print(new_risk["Risk"].sum(),sum(solution.variables[0]))

In [ ]:
algorithm = NSGAII(SensorProblem(
    riskmap,
    [
        {
            "info_col": "csoInfo",
            "risk_name": "csoRisk",
            "risk_method": csoRisk.add_duration_risk,
            "distance_scaling": linear_dist_decrease,
        }
    ],
    2_000
))

In [ ]:
solutions = [s for s in algorithm.result if s.feasible]
solutions

In [ ]:
sensorlocs = riskmap.get_risk_map()[unique(
    nondominated(algorithm.result))[0].variables[0]]
sensorlocs

In [ ]:
print(riskmap.get_risk_map().shape)
riskmap.get_risk_map().explore(column="Risk")

### PyMoo

In [ ]:
from pymoo.core.problem import ElementwiseProblem

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.operators.crossover.binx import BinomialCrossover
from pymoo.operators.selection.tournament import TournamentSelection
from pymoo.optimize import minimize
from pymoo.termination.ftol import MultiObjectiveSpaceTermination
from pymoo.termination.robust import RobustTermination

In [ ]:
class MyProblem(ElementwiseProblem):

    def __init__(self, riskmap: RiskMap.RiskMap, riskList: list[dict], max_dist_sensor: float):

        l = len(riskmap.get_risk_map())

        super().__init__(n_var=l, n_obj=1, n_ieq_constr=1,)
        self.riskmap = riskmap
        self.max_dist_sensor = max_dist_sensor
        self.riskList = riskList

    def risk_after_coverage(self, x) -> float:
        sensorLocs = self.riskmap.get_risk_map()[x]
        riskList2 = []
        for item in self.riskList:
            riskList2.append(item.copy())
            riskList2[-1]["ids_to_ignore"] = [thing[0] for row in sensorLocs[item["info_col"]].to_list(
            ) for thing in row if thing[1] < self.max_dist_sensor]

        new_risk = self.riskmap.add_all_risk(riskList2,
                                             inplace=False)
        return new_risk["Risk"].sum()/len(new_risk)

    def _evaluate(self, x, out, *args, **kwargs):
        out["F"] = self.risk_after_coverage(x)
        out["G"] = sum(x) - N_SENSORS

In [ ]:
algorithm = NSGA2(
    pop_size=100,
    n_offsprings=100,
    sampling=BinaryRandomSampling(),
    crossover=BinomialCrossover(),
    mutation=BitflipMutation(prob=0.2),
    eliminate_duplicates=True
)

In [ ]:
problem = MyProblem(
    riskmap,
    [
        {
            "info_col": "csoInfo",
            "risk_name": "csoRisk",
            "risk_method": csoRisk.add_duration_risk,
            "distance_scaling": linear_dist_decrease,
        },
    ],
    2_000
)

In [ ]:
res = minimize(
    problem,
    algorithm,
    termination=RobustTermination(MultiObjectiveSpaceTermination(tol=0.5), 20),
    # save_history=True,
    verbose=True,
)

In [ ]:
print(len(res.X))
sol_number = 3

In [ ]:
sensors = riskmap.get_risk_map()[res.X[sol_number]]

In [ ]:
# Obj from MOO
problem.risk_after_coverage(res.X[sol_number])

In [ ]:
# Obj from Greedy
problem.risk_after_coverage(riskmap.get_risk_map().geometry.apply(
    lambda x: x in history[-1][0].geometry))

In [ ]:
# Obj without any sensors
riskmap.get_risk_map()["Risk"].sum()/len(riskmap.get_risk_map())

In [ ]:
m = rivers.explore()
sensors.explore(m=m, marker_kwds={"radius": 5}, color="red")
history[-1][0].explore(m=m, marker_kwds={"radius": 5}, color="blue")
riskmap.get_risk_map().explore(m=m, column="Risk")